In [1]:
import spacy
import httpx
import time
import re
import os
import pandas as pd
from rapidfuzz import fuzz
from dotenv import load_dotenv
from transformers import pipeline

c:\Users\adamd\miniconda3\envs\movie-sentiments\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
TMDB_API_KEY = os.getenv("TMDB_READ_ACCESS_TOKEN")
 
# check TMDb key is loaded
if not TMDB_API_KEY:
    print("WARNING: TMDB_READ_ACCESS_TOKEN not found in .env")
else:
    print("TMDb API key loaded")

TMDb API key loaded


In [3]:
# load SpaCy model (using the transformer model, for more info see https://spacy.io/models/en)
nlp = spacy.load("en_core_web_trf")
print(f"SpaCy model loaded: {nlp.meta['name']}")

SpaCy model loaded: core_web_trf


In [4]:
# API docs: https://developer.themoviedb.org/reference/search-multi
TMDB_SEARCH_URL = "https://api.themoviedb.org/3/search/multi"

#function to query the movie database API using a WORK_OF_ART string
#using httpx instead of requests library with an eye on using async via httpx down the line
def search_tmdb(WORK_OF_ART):
    #search the movie database using WORK_OF_ART, returns the most popular matching movie/tv show or nothing
    headers = {
        "Authorization": f"Bearer {TMDB_API_KEY}",
        "accept": "application/json",
    }
    #send GET request to API, with a 10s timeout before raising error
    resp = httpx.get(TMDB_SEARCH_URL, headers=headers, params={"query": WORK_OF_ART}, timeout=10)
    #raise error code
    resp.raise_for_status()

    #filter the API response on movie and tv show - the search/multi endpoint also includes people, but this isnt relevant 
    results = [r for r in resp.json().get("results", []) if r.get("media_type") in ("movie", "tv")]
    if not results:
        return None

    #select the top result based on popularity (this is defined by themoviedb here:https://starlight.mintlify.app/api-reference/popularandtrending)
    top = results[0]

    #return result as dictionary
    return {
        "tmdb_id": top.get("id"),
        #get title or name, depending on movie or tv show
        "tmdb_title": top.get("title") or top.get("name", ""),
        "media_type": top.get("media_type"),
        #get release date or first air date depending on movie or tv show
        "release_date": top.get("release_date") or top.get("first_air_date", ""),
        #get average vote data for some extra context, might use it later
        "vote_average": top.get("vote_average", 0),
    }

In [5]:
#link NER WORK_OF_ART results against the movie database (using a personal API TMDB_READ_ACCESS_TOKEN set in .env file)
def extract_entity_and_link(text, nlp_model, entity_type):
    doc = nlp_model(text)
    #loop through every entity that the NER model found, keep only WORK_OF_ART entities, 
    # and extract their text. use set to remove duplicates, and convert back to list.
    #using ent.label_ to get human readable label
    works = list(set(ent.text for ent in doc.ents if ent.label_ == entity_type))

    linked = []
    for title in works:
        result = search_tmdb(title)
        #sleep for courtesy delay. Docs says rate limit of 40 per second so this is well within range
        time.sleep(0.10)
        if result:
            result["extracted_entity"] = title
            linked.append(result)
    return linked

In [6]:
# run NER and moviedb API call then storing results in df
df = pd.read_csv("../data/backfill_watched_gold_labelled.csv")
positives = df[df["gold_label"] == 1]

results = []
for idx, row in positives.iterrows():
    for match in extract_entity_and_link(row["text"], nlp, "WORK_OF_ART"):
        match["post_text"] = row["text"]
        results.append(match)

linked_df = pd.DataFrame(results)
print(f"\nTotal titles found in posts: {len(linked_df)}")
print(f"Unique titles found in posts: {linked_df['tmdb_title'].nunique()}")
linked_df.head(10)


Total titles found in posts: 115
Unique titles found in posts: 95


,tmdb_id,tmdb_title,media_type,release_date,vote_average,extracted_entity,post_text
0,416154,CarousHELL,movie,2016-09-02,4.364,Caroushell,Yesterday I watched a movie called Caroushell ...
1,1558839,Frogman Returns,movie,2025-10-19,0.000,Frogman,Yesterday I watched a movie called Caroushell ...
2,49018,Insidious,movie,2011-03-31,6.953,Insidious,The latest Insidious movie gave me everything ...
3,10195,Thor,movie,2011-04-21,6.800,Thor,"The thing is, many of them have themes! Often ..."
4,569094,Spider-Man: Across the Spider-Verse,movie,2023-05-31,8.346,Spiderverse,This is one of the many things the animated Sp...
5,1940,Entourage,tv,2004-07-18,7.366,ENTOURAGE,I won’t say ENTOURAGE holds up particularly we...
6,88396,The Falcon and the Winter Soldier,tv,2021-03-19,7.590,The Winter Soldier,[Minor Spoiler for a 12 year old movie]\n\nOne...
7,13939,Death Wish,movie,1974-07-24,6.931,Death Wish,"Man, I love the Death Wish movies. Their messa..."
8,24873,Death Wish 3,movie,1985-11-01,6.218,Death Wish III,"Man, I love the Death Wish movies. Their messa..."
9,858017,I Saw the TV Glow,movie,2024-05-03,6.198,I Saw the TV Glow,This is a major way the movie ends up overstuf...


In [7]:
#load sentiment model (using twitter-roberta-base-sentiment-latest)
sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=-1,  #run on cpu
)

# test
print(sentiment_model("That movie was absolutely incredible"))
print(sentiment_model("Worst film I have ever seen"))
print(sentiment_model("I watched it, it was okay I guess"))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9662.85it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'label': 'positive', 'score': 0.9652649760246277}]
[{'label': 'negative', 'score': 0.9403607249259949}]
[{'label': 'positive', 'score': 0.6957908868789673}]


In [9]:
#batched approach to running sentiment analysis on posts which have a movie/tv show title
results = sentiment_model(linked_df["post_text"].tolist(), batch_size=32)

linked_df["sentiment"] = [r["label"] for r in results]
linked_df["sentiment_probability"] = [round(r["score"], 3) for r in results]

print(linked_df.head(20).to_string())

    tmdb_id                            tmdb_title media_type release_date  vote_average                    extracted_entity                                                                                                                                                                                                                                                                                                  post_text sentiment  sentiment_probability
0    416154                            CarousHELL      movie   2016-09-02         4.364                          Caroushell                                                                     Yesterday I watched a movie called Caroushell (brought to my attention by my friend who also watched it) and it was an Experience. Today I am watching a movie called Frogman.\n\n"Are you saying frogman fucks?" Incredible dialogue.  positive                  0.835
1   1558839                       Frogman Returns      movie   2025-10-19         0.000 

In [10]:
#aggregate analysis
agg_analysis = linked_df.groupby("tmdb_title").agg(
    post_mention_count=("tmdb_title", "count"),
    #lambda functions to count case when sentiment occurs in each row
    positive_count=("sentiment", lambda x: (x == "positive").sum()),
    neutral_count=("sentiment", lambda x: (x == "neutral").sum()),
    negative_count=("sentiment", lambda x: (x == "negative").sum()),
    #"first" to get the first value of each group, since each title will have the same media type
    media_type=("media_type", "first"),
    release_date=("release_date", "first")

).sort_values("post_mention_count", ascending=False)

print(agg_analysis.head(20).to_string())

                                                                                            post_mention_count  positive_count  neutral_count  negative_count media_type release_date
tmdb_title                                                                                                                                                                           
Coyote vs. Acme                                                                                             10              10              0               0      movie   2026-08-20
New Looney Tunes                                                                                             4               4              0               0         tv   2015-09-21
The Rocky Horror Picture Show                                                                                3               0              0               3      movie   1975-08-14
Insidious                                                                                 